# Phase 3 — Fine-tuning DistilGPT-2 on the cleaned AG News corpus

This notebook closes the loop of the project: the C++ pipeline (sequential /
OpenMP / MPI / hybrid) turns raw AG News into a **clean** corpus
(`results/clean_full.txt`, one cleaned document per line); here we fine-tune
**DistilGPT-2** on that corpus.

**Run on Google Colab with a GPU runtime** (`Runtime → Change runtime type →
T4 GPU`). The cleaned file produced by *any* of the four pipeline versions is
byte-identical, so it does not matter which one generated it.

Steps: install deps → upload the cleaned corpus → tokenize → fine-tune →
generate sample text → save the model.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Provide the cleaned corpus

Upload `results/clean_full.txt` (or `clean_10k.txt` for a quick run) produced by
the C++ pipeline. If the upload widget is inconvenient, you can instead mount
Google Drive and point `CORPUS_PATH` at the file.

In [ ]:
import os

CORPUS_PATH = "clean_full.txt"  # change to clean_10k.txt for a fast demo

if not os.path.exists(CORPUS_PATH):
    try:
        from google.colab import files
        print("Select your cleaned corpus file (e.g. clean_full.txt) ...")
        uploaded = files.upload()
        CORPUS_PATH = next(iter(uploaded))
    except Exception as e:
        raise FileNotFoundError(
            f"{CORPUS_PATH} not found and upload failed ({e}). "
            "Upload the file or mount Drive and set CORPUS_PATH."
        )

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    lines = [ln.strip() for ln in f if ln.strip()]
print(f"Loaded {len(lines):,} cleaned documents from {CORPUS_PATH}")
print("Example:", lines[0][:200])

## 3. Build a Hugging Face Dataset and tokenize

We use DistilGPT-2's own **BPE** tokenizer — this is the "real tokenizer" that
the project's whitespace Stage-4 tokenizer was a stand-in for. We pack the
documents into fixed-length blocks (standard causal-LM training).

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilgpt2"
BLOCK_SIZE = 128

raw_ds = Dataset.from_dict({"text": lines})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

def tokenize_fn(batch):
    return tokenizer(batch["text"])

tokenized = raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

def group_texts(examples):
    # Concatenate then split into BLOCK_SIZE chunks.
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_len = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_ds = tokenized.map(group_texts, batched=True)

# Small held-out split so step 5 can report a validation perplexity.
split = lm_ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds, eval_ds)

## 4. Fine-tune DistilGPT-2

One epoch is enough to demonstrate the model adapting to the news-text
distribution. Increase `num_train_epochs` for a stronger fit.

In [ ]:
import inspect
from transformers import (
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Build the desired training arguments, then keep ONLY the ones this installed
# version of `transformers` actually accepts. The TrainingArguments signature
# changes between versions (e.g. newer releases dropped `overwrite_output_dir`),
# so filtering by the real signature makes this cell robust to any version.
desired = dict(
    output_dir="distilgpt2-agnews",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
supported = set(inspect.signature(TrainingArguments.__init__).parameters)
args = TrainingArguments(**{k: v for k, v in desired.items() if k in supported})

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
)

trainer.train()

## 5. Evaluate: perplexity + sample generation

In [ ]:
import math

metrics = trainer.evaluate()
if "eval_loss" in metrics:
    print(f"Eval loss: {metrics['eval_loss']:.4f}  |  "
          f"Perplexity: {math.exp(metrics['eval_loss']):.2f}")

from transformers import pipeline

gen = pipeline("text-generation", model=model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1)

for prompt in ["the stock market", "the government announced", "scientists have discovered"]:
    out = gen(prompt, max_new_tokens=40, num_return_sequences=1,
              do_sample=True, top_k=50, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
    print("PROMPT:", prompt)
    print(" ->", out[0]["generated_text"].replace("\n", " "))
    print()

## 6. Save the fine-tuned model

In [ ]:
save_dir = "distilgpt2-agnews-final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved fine-tuned model to:", save_dir)

# Zip the model folder and download it to your computer.
import shutil
from google.colab import files

zip_path = shutil.make_archive("distilgpt2-agnews-final", "zip", save_dir)
print("Created archive:", zip_path)
files.download(zip_path)   # browser download prompt appears

---
### How this ties back to the C++ pipeline

The corpus uploaded in step 2 is the output of the project's preprocessing
pipeline — Stage 1 (clean) → Stage 2 (quality filter) → Stage 3 (dedup) →
Stage 4 (tokenize/count). Removing junk, low-quality, and duplicate documents
*before* training is exactly what makes fine-tuning cheaper and the model
better, which is the whole motivation for parallelizing the pipeline
(Sequential → OpenMP → MPI → Hybrid) so it can scale to corpora far larger than
AG News.